# HyperParameter Tunning (LGBM)

In [ ]:
# Imports

import optuna
import pandas as pd
from lightgbm import LGBMClassifier
from sklearn.preprocessing import OrdinalEncoder
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score

In [ ]:
# Data Load

df = pd.read_csv(r"train.csv")

df

,id,Age,Flight Distance,Inflight wifi service,Departure/Arrival time convenient,Ease of Online booking,Gate location,Food and drink,Online boarding,Seat comfort,...,Baggage handling,Checkin service,Cleanliness,Departure Delay in Minutes,Arrival Delay in Minutes,Gender,Customer Type,Type of Travel,Class,satisfaction
0,0,36,694,4,4,4,3,1,4,1,...,4,3,1,0,0.0,Female,Loyal Customer,Personal Travel,Eco,False
1,1,56,651,5,5,5,5,5,4,4,...,5,4,3,0,0.0,Male,Loyal Customer,Business travel,Business,True
2,2,31,1371,3,4,2,5,3,3,3,...,4,5,3,0,0.0,Female,Loyal Customer,Personal Travel,Eco,False
3,3,10,1616,3,5,3,3,3,3,3,...,3,4,3,0,0.0,Male,Loyal Customer,Personal Travel,Eco,False
4,4,23,481,3,4,3,4,5,3,5,...,5,3,5,5,0.0,Female,disloyal Customer,Business travel,Eco,False
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
699630,699630,57,612,2,1,1,1,1,3,3,...,2,2,4,0,0.0,Female,Loyal Customer,Business travel,Eco,False
699631,699631,30,862,2,2,2,3,4,2,4,...,4,2,4,24,34.0,Female,disloyal Customer,Business travel,Eco,False
699632,699632,49,3148,5,5,5,5,4,4,4,...,5,5,5,0,0.0,Male,Loyal Customer,Business travel,Business,True
699633,699633,61,679,5,2,2,2,5,4,4,...,5,5,1,0,0.0,Female,Loyal Customer,Business travel,Eco,True


In [ ]:
# Preprocess

def get_X_y(df) :

    ranges = [0 , 10 , 20 , 30 , 40 , 50 , 60 , 70 , 80 , 90 , 100 , 200 , 300 , 400 , 500]

    j = 1

    for i in range(len(ranges) - 1):

        condition1 = (
        (df["Departure Delay in Minutes"] > ranges[i]) &
        (df["Departure Delay in Minutes"] <= ranges[j]) &
        (df["Arrival Delay in Minutes"].isnull())
        )

        condition2 = (
        (df["Departure Delay in Minutes"] > ranges[i]) &
        (df["Departure Delay in Minutes"] <= ranges[j])
        )

        col = "Arrival Delay in Minutes"

        fill = float((df["Departure Delay in Minutes"][condition2]).mean())

        fill = round(fill , 2)

        df.loc[condition1 , col] = fill

        j = j + 1

    df.loc[df["Arrival Delay in Minutes"].isnull() , "Arrival Delay in Minutes"] = 0

    df["Total Delay"] = df["Departure Delay in Minutes"] + df["Arrival Delay in Minutes"]

    # df["Delay Diff"] = df["Departure Delay in Minutes"] - df["Arrival Delay in Minutes"]

    # df["Overall Rating"] = df["Inflight wifi service"]  + df["Baggage handling"] + df["On-board service"] + df["Leg room service"] + df["Inflight entertainment"] + df["Departure/Arrival time convenient"] + df["Ease of Online booking"] + df["Gate location"] + df["Food and drink"] + df["Online boarding"] + df["Seat comfort"] + df["Checkin service"] + df["Cleanliness"]

    # df["distance * Rating"] = df["Flight Distance"] * df["Overall Rating"]

    # df["distance / Rating"] = df["Flight Distance"] / df["Overall Rating"]

    # df["Inflight_Rating"] = df["Inflight wifi service"] + df["Food and drink"] + df["Seat comfort"] + df[ "Inflight entertainment"] + df["On-board service"] + df["Leg room service"]

    # df["Outflight_Rating"] = df["Departure/Arrival time convenient"] + df["Ease of Online booking"] + df["Gate location"] + df["Online boarding"] + df["Baggage handling"] + df["Checkin service"]

    df = df.drop(["Arrival Delay in Minutes" , "Departure Delay in Minutes" ] , axis = 1)

    X = df.drop(["satisfaction" , "id"] , axis = 1)
    y = df[["satisfaction"]]

    y = OrdinalEncoder().fit_transform(y)
    y = (pd.Series(y.flatten())).astype(int)

    return X , y

In [ ]:
# Data Prep

X_train , y_train = get_X_y(df)

X_train , X_test , y_train , y_test = train_test_split(
    X_train ,
    y_train ,
    test_size = 0.2 ,
    random_state = 42 ,
    stratify = y_train
)

cat_cols = X_train.select_dtypes(include = ["string" , "object" , "category"]).columns

for col in cat_cols:
    X_train[col] = X_train[col].astype("category")
    X_test[col] = X_test[col].astype("category")

In [ ]:
# 1st Try

def objective(trial) :

  param = {

      "lgbm__n_estimators" : trial.suggest_int(
          "n_estimator" ,
          500 ,
          5000
      ) ,

      "lgbm__learning_rate" : trial.suggest_float(
          "learning_rate" ,
          0.005 ,
          0.15 ,
          log = True
      ) ,

      "lgbm__num_leaves" : trial.suggest_int(
          "num_leaves" ,
          16 ,
          256
      ) ,

      "lgbm__max_depth" : trial.suggest_int(
          "max_depth" ,
          3 ,
          16
      ) ,

      "lgbm__min_child_samples" : trial.suggest_int(
          "min_child_samples" ,
          10 ,
          200
      ) ,

      "lgbm__min_child_weight" : trial.suggest_float(
          "min_child_weight" ,
          1e-4 ,
          10 ,
          log = True
      ) ,

      "subsample" : trial.suggest_float(
          "subsample" ,
          0.60 ,
          1.00
      ) ,

      "subsample_freq" : trial.suggest_int(
          "subsample_freq" ,
          1 ,
          10
      ) ,

      "colsample_bytree" : trial.suggest_float(
          "colsample_bytree" ,
          0.50 ,
          1.00
      ) ,

      "colsample_bynode" : trial.suggest_float(
          "colsample_bynode" ,
          0.50 ,
          1.00
      ) ,

      "extra_trees" : trial.suggest_categorical(
          "extra_trees" ,
          [False , True]
      ) ,

      "reg_alpha" : trial.suggest_float(
          "reg_alpha" ,
          1e-8 ,
          100 ,
          log = True
      ) ,

      "reg_lambda" : trial.suggest_float(
          "reg_lambda" ,
          1e-8 ,
          100 ,
          log = True
      ) ,

      "min_split_gain" : trial.suggest_float(
          "min_split_gain" ,
          0 ,
          5
      ) ,

      "path_smooth" : trial.suggest_float(
          "path_smooth" ,
          0 ,
          100
      ) ,

      "max_bin" : trial.suggest_int(
          "max_bin" ,
          63 ,
          511
      ) ,

      "min_data_in_bin" : trial.suggest_int(
          "min_data_in_bin" ,
          3 ,
          100
      ) ,

      "cat_smooth" : trial.suggest_float(
          "cat_smooth" ,
          1 ,
          100 ,
          log = True
      ) ,

      "cat_l2" : trial.suggest_float(
          "cat_l2" ,
          1e-3 ,
          100 ,
          log = True
      ) ,

      "max_cat_threshold" : trial.suggest_int(
          "max_cat_threshold" ,
          16 ,
          128
      ) ,

      "max_cat_to_onehot" : trial.suggest_int(
          "max_cat_to_onehot" ,
          2 ,
          16
      ) ,

      "categorical_feature" : "auto" ,
      "objective" : "binary" ,
      "metric" : "auc" ,
      "boosting_type" : "gbdt" ,
      "importance_type" : "gain" ,
      "random_state" : 42 ,
      "verbosity" : -1 ,
      "n_jobs" : -1

  }

  trial_model = LGBMClassifier(
      **param
  )

  trial_model.fit(
      X_train ,
      y_train
  )

  y_proba = trial_model.predict_proba(X_test)[ : , 1]

  auc = roc_auc_score(y_test, y_proba)

  return auc


study = optuna.create_study(
    study_name = "STUDY 1",
    direction = "maximize"
)

study.optimize(
    objective,
    n_trials = 200
)

print("Best ROC-AUC:", study.best_value)
print("Best Parameters:")
print(study.best_params)

[I 2026-10-17 11:16:41,160] A new study created in memory with name: STUDY 1
/usr/local/lib/python3.13/dist-packages/lightgbm/basic.py:2137: UserWarning: categorical_feature keyword has been found in `params` and will be ignored.
Please use categorical_feature argument of the Dataset constructor to pass this parameter.
  _log_warning(
/usr/local/lib/python3.13/dist-packages/lightgbm/basic.py:2159: UserWarning: categorical_feature in param dict is overridden.
  _log_warning(f"{cat_alias} in param dict is overridden.")
[I 2026-10-17 11:16:51,163] Trial 0 finished with value: 0.9494537106533092 and parameters: {'n_estimator': 698, 'learning_rate': 0.06794562553941749, 'num_leaves': 174, 'max_depth': 13, 'min_child_samples': 124, 'min_child_weight': 0.0026803379047477737, 'subsample': 0.7792944017875123, 'subsample_freq': 3, 'colsample_bytree': 0.9208319817991135, 'colsample_bynode': 0.6401598852763382, 'extra_trees': True, 'reg_alpha': 2.452525558891421e-05, 'reg_lambda': 1.38170760914088

Best ROC-AUC: 0.957090388614289
Best Parameters:
{'n_estimator': 1459, 'learning_rate': 0.08343741504913935, 'num_leaves': 181, 'max_depth': 6, 'min_child_samples': 79, 'min_child_weight': 0.001870299624353973, 'subsample': 0.8817434745029977, 'subsample_freq': 5, 'colsample_bytree': 0.9676512427147542, 'colsample_bynode': 0.7256650443257682, 'extra_trees': False, 'reg_alpha': 9.55188266573322e-06, 'reg_lambda': 2.6381557025996975e-08, 'min_split_gain': 0.018237667650837786, 'path_smooth': 36.55798774092856, 'max_bin': 490, 'min_data_in_bin': 47, 'cat_smooth': 5.273632681271112, 'cat_l2': 0.2943297433072277, 'max_cat_threshold': 103, 'max_cat_to_onehot': 9}


In [ ]:
# 1st Try

def objective(trial) :

  param = {

      "lgbm__n_estimators" : trial.suggest_int(
          "n_estimator" ,
          500 ,
          5000
      ) ,

      "lgbm__learning_rate" : trial.suggest_float(
          "learning_rate" ,
          0.005 ,
          0.15 ,
          log = True
      ) ,

      "lgbm__num_leaves" : trial.suggest_int(
          "num_leaves" ,
          16 ,
          256
      ) ,

      "lgbm__max_depth" : trial.suggest_int(
          "max_depth" ,
          3 ,
          16
      ) ,

      "lgbm__min_child_samples" : trial.suggest_int(
          "min_child_samples" ,
          10 ,
          200
      ) ,

      "lgbm__min_child_weight" : trial.suggest_float(
          "min_child_weight" ,
          1e-4 ,
          10 ,
          log = True
      ) ,

      "subsample" : trial.suggest_float(
          "subsample" ,
          0.60 ,
          1.00
      ) ,

      "subsample_freq" : trial.suggest_int(
          "subsample_freq" ,
          1 ,
          10
      ) ,

      "colsample_bytree" : trial.suggest_float(
          "colsample_bytree" ,
          0.50 ,
          1.00
      ) ,

      "colsample_bynode" : trial.suggest_float(
          "colsample_bynode" ,
          0.50 ,
          1.00
      ) ,

      "extra_trees" : trial.suggest_categorical(
          "extra_trees" ,
          [False , True]
      ) ,

      "reg_alpha" : trial.suggest_float(
          "reg_alpha" ,
          1e-8 ,
          100 ,
          log = True
      ) ,

      "reg_lambda" : trial.suggest_float(
          "reg_lambda" ,
          1e-8 ,
          100 ,
          log = True
      ) ,

      "min_split_gain" : trial.suggest_float(
          "min_split_gain" ,
          0 ,
          5
      ) ,

      "path_smooth" : trial.suggest_float(
          "path_smooth" ,
          0 ,
          100
      ) ,

      "max_bin" : trial.suggest_int(
          "max_bin" ,
          63 ,
          511
      ) ,

      "min_data_in_bin" : trial.suggest_int(
          "min_data_in_bin" ,
          3 ,
          100
      ) ,

      "cat_smooth" : trial.suggest_float(
          "cat_smooth" ,
          1 ,
          100 ,
          log = True
      ) ,

      "cat_l2" : trial.suggest_float(
          "cat_l2" ,
          1e-3 ,
          100 ,
          log = True
      ) ,

      "max_cat_threshold" : trial.suggest_int(
          "max_cat_threshold" ,
          16 ,
          128
      ) ,

      "max_cat_to_onehot" : trial.suggest_int(
          "max_cat_to_onehot" ,
          2 ,
          16
      ) ,

      "categorical_feature" : "auto" ,
      "objective" : "binary" ,
      "metric" : "auc" ,
      "boosting_type" : "gbdt" ,
      "importance_type" : "gain" ,
      "random_state" : 42 ,
      "verbosity" : -1 ,
      "n_jobs" : -1

  }

  trial_model = LGBMClassifier(
      **param
  )

  trial_model.fit(
      X_train ,
      y_train
  )

  y_proba = trial_model.predict_proba(X_test)[ : , 1]

  auc = roc_auc_score(y_test, y_proba)

  return auc


study = optuna.create_study(
    study_name = "STUDY 1",
    direction = "maximize"
)

study.optimize(
    objective,
    n_trials = 200
)

print("Best ROC-AUC:", study.best_value)
print("Best Parameters:")
print(study.best_params)

[I 2026-10-17 10:43:37,701] A new study created in memory with name: STUDY 1
/usr/local/lib/python3.13/dist-packages/lightgbm/basic.py:2137: UserWarning: categorical_feature keyword has been found in `params` and will be ignored.
Please use categorical_feature argument of the Dataset constructor to pass this parameter.
  _log_warning(
/usr/local/lib/python3.13/dist-packages/lightgbm/basic.py:2159: UserWarning: categorical_feature in param dict is overridden.
  _log_warning(f"{cat_alias} in param dict is overridden.")
[I 2026-10-17 10:43:52,327] Trial 0 finished with value: 0.9553959008360156 and parameters: {'n_estimator': 2048, 'learning_rate': 0.05379372319904947, 'num_leaves': 50, 'max_depth': 12, 'min_child_samples': 70, 'min_child_weight': 0.20609977897269388, 'subsample': 0.7006475868661192, 'subsample_freq': 8, 'colsample_bytree': 0.5320291029071285, 'colsample_bynode': 0.5036569433874842, 'extra_trees': False, 'reg_alpha': 6.742066326011346, 'reg_lambda': 1.3174933635289265e-08

Best ROC-AUC: 0.9572494049962101
Best Parameters:
{'n_estimator': 2257, 'learning_rate': 0.01400207637043854, 'num_leaves': 81, 'max_depth': 5, 'min_child_samples': 194, 'min_child_weight': 0.00023547227584805327, 'subsample': 0.8078157370581776, 'subsample_freq': 4, 'colsample_bytree': 0.8429699233588869, 'colsample_bynode': 0.8422750811378343, 'extra_trees': False, 'reg_alpha': 6.7879676933594535, 'reg_lambda': 5.356757716738945e-08, 'min_split_gain': 0.0466383349781442, 'path_smooth': 6.72941933379196, 'max_bin': 504, 'min_data_in_bin': 78, 'cat_smooth': 2.6421734086304727, 'cat_l2': 0.0011519382614044997, 'max_cat_threshold': 66, 'max_cat_to_onehot': 3}
